In [9]:
import neurokit2 as nk
import numpy as np
import pandas as pd
import tqdm
import matplotlib.pyplot as plt
 
import sys
sys.path.insert(1, '../src')

from data_loader import cleaned_data
from data_processing import pad_data
import ast

In [10]:
data = cleaned_data()
padded_data = pad_data(data)
print(padded_data.shape)

padding data: 100%|██████████| 21799/21799 [17:36<00:00, 20.64it/s]


(21799, 1100, 12)


In [11]:
# filepath = '../data/processed/cleaned_data.csv'
# d = np.array(data)
# pd.DataFrame(d.reshape(d.shape[0], -1)).to_csv(filepath, index=False)

print(len(padded_data[0][0]))

12


In [13]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from PQRST_extraction import process_sample

if __name__ == "__main__":
    
    futures = []
    
    with ProcessPoolExecutor() as executor:
        for i, (padded_sample, sample) in enumerate(zip(padded_data, data)):
            # plt.plot(padded_sample[:,0], color = "black") # each sample contains all 12 leads. process 12 leads separately
            for j in range(len(padded_sample[0])):
                futures.append(executor.submit(process_sample, padded_sample[:,j], sample[:,j], i, j))
            
        with tqdm.tqdm(total=len(futures), desc="extracting peaks") as progress:
            sample_peaks = []
            for future in as_completed(futures):
                sample_peaks.append(future.result())
                progress.update(1)
            
    print("Peaks Extracted")


extracting peaks: 100%|██████████| 261588/261588 [1:19:04<00:00, 55.14it/s]  


Peaks Extracted


In [14]:
sample_peaks = []
for future in futures:
    sample_peaks.append(future.result())


In [15]:
print(type(sample_peaks[0]))
for i in range(5):
    print(sample_peaks[i])

<class 'dict'>
{'ECG_R_Peaks': [25, 117, 210, 301, 395, 490, 585, 681, 775, 867, 962], 'ECG_Q_Peaks': [15, 113, 201, 298, 391, 483, 575, 671, 767, 862, 955], 'ECG_S_Peaks': [38, 128, 222, 314, 409, 500, 598, 692, 789, 884, 976], 'ECG_T_Peaks': [43, 136, 235, 328, 421, 502, 600, 707, 802, 893, 989], 'ECG_P_Peaks': [100, 193, 284, 377, 470, 565, 667, 758, 854, 946]}
{'ECG_R_Peaks': [22, 115, 208, 300, 393, 487, 583, 679, 773, 864, 961], 'ECG_Q_Peaks': [19, 108, 204, 298, 389, 483, 580, 671, 769, 859, 957], 'ECG_S_Peaks': [24, 117, 210, 302, 395, 490, 585, 681, 775, 866, 963], 'ECG_T_Peaks': [42, 129, 220, 323, 412, 498, 596, 703, 797, 882, 988], 'ECG_P_Peaks': [100, 193, 289, 373, 470, 564, 667, 753, 846, 947]}
{'ECG_R_Peaks': [22, 114, 207, 300, 393, 487, 582, 678, 773, 864, 960], 'ECG_Q_Peaks': [14, 112, 200, 297, 385, 485, 580, 671, 770, 859, 957], 'ECG_S_Peaks': [24, 117, 209, 302, 395, 489, 585, 681, 775, 866, 963], 'ECG_T_Peaks': [43, 129, 220, 316, 412, 493, 608, 684, 796, 871, 96

In [16]:
np_peaks = np.array(sample_peaks, dtype=dict)
peaks = np.reshape(np_peaks, (-1, 12))
print(len(peaks))
print(peaks[0][0])

21799
{'ECG_R_Peaks': [25, 117, 210, 301, 395, 490, 585, 681, 775, 867, 962], 'ECG_Q_Peaks': [15, 113, 201, 298, 391, 483, 575, 671, 767, 862, 955], 'ECG_S_Peaks': [38, 128, 222, 314, 409, 500, 598, 692, 789, 884, 976], 'ECG_T_Peaks': [43, 136, 235, 328, 421, 502, 600, 707, 802, 893, 989], 'ECG_P_Peaks': [100, 193, 284, 377, 470, 565, 667, 758, 854, 946]}


In [17]:
import json_format

list_peaks = peaks.tolist()

fp = "../data/processed/ecg_peaks.json"
json_peaks = json_format.to_json(list_peaks)
with open(fp, 'w') as f:
    f.write(json_peaks) # default=lambda x: list(x) if isinstance(x, tuple) else str(x)
